<h3>Importing my development version of ShadowGrouping</h3>

In [1]:
import sys, numpy as np, time, pickle
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial
import matplotlib.pyplot as plt
from IPython.display import Image, display
from os import mkdir
from os.path import isdir, isfile

# adding ShadowGrouping package to the system path
# SG_package_path = r"G:\\My Drive\\Work\\Research\\Numerics\\ShadowGrouping Code\\"
#SG_package_path = r"C:\Users\aebad\shadowgrouping\\"
SG_package_path = r"C:\Users\Abbas\Priori\\"
sys.path.insert(0, SG_package_path)
#savename = ".txt" # insert {mapping_name}
savepath = "generated_data/random_Hamiltonians/"
# defining path where Hamiltonians are stored
folder_Hamiltonians = SG_package_path + "Hamiltonians_v2\\"

In [2]:
from shadowgrouping_v2.measurement_schemes import L1_sampler, Derandomization
#from shadowgrouping.measurement_schemes import Shadow_Grouping
from shadowgrouping.measurement_schemes_Shadowupdate import Priori, Posteriori, Shadow_Grouping, Best_scheme_given_pool
from shadowgrouping_v2.weight_functions import Bernstein_bound
from shadowgrouping_v2.energy_estimator import Energy_estimator, StateSampler, Sign_estimator
from shadowgrouping_v2.helper_functions import (
    setting_to_str, char_to_int, int_to_char, settings_to_dict, prepare_settings_for_numba, gaussian_weights, gaussian_random_paulis, uniform_weights, uniform_random_paulis,
    sample_obs_from_setting, setting_to_str, setting_to_obs_form, sample_obs_batch_from_setting_batch, sample_obs_batch_from_setting_batch_numba, bootstrap_rmse)
from shadowgrouping_v2.hamiltonian import get_pauli_list, get_groundstate, mappings, load_pauli_list, apply_Hamiltonian_to_state
from shadowgrouping_v2.guarantees import (
    get_epsilon_Bernstein, get_epsilon_Bernstein_no_restricted_validity, 
    get_epsilon_Bernstein_tighter, get_epsilon_Bernstein_tighter_no_restricted_validity, 
    get_epsilon_Bernstein_scalar, get_epsilon_Bernstein_scalar_no_restricted_validity,
    get_epsilon_Bernstein_scalar_tighter, get_epsilon_Bernstein_scalar_tighter_no_restricted_validity, 
    get_epsilon_Hoeffding_scalar, get_epsilon_Hoeffding_scalar_tighter, get_epsilon_Chebyshev_scalar, 
    get_epsilon_Chebyshev_scalar_tighter, get_epsilon_single_Hoeffding_plus_union_bound,
    get_epsilon_Chebyshev_scalar_numba, get_epsilon_Chebyshev_scalar_tighter_numba,
    get_epsilon_Hoeffding_scalar_numba, get_epsilon_Hoeffding_scalar_tighter_numba)
from guarantees.exact_covariances import covariance_matrix
from guarantees.guarantees import (get_epsilon_Chebyshev_scalar_tighter_numba, get_epsilon_Chebyshev_scalar_tightest_numba)
from shadowgrouping.poolgenerators import Priori_Pool , OGM_Pool, AEQUO_Pool, AEQUO_Pool_weightbased, OGM_NPBC

C:\Users\Abbas\AppData\Local\Temp\ipykernel_18676\2440937489.py:9: NatureDeprecationWarning: The qiskit_nature.drivers.second_quantization package is deprecated as of version 0.5.0 and will be removed no sooner than 3 months after the release. Instead use the qiskit_nature.second_q.drivers package.
  from shadowgrouping_v2.hamiltonian import get_pauli_list, get_groundstate, mappings, load_pauli_list, apply_Hamiltonian_to_state


<h3>Choosing molecule, basis set and fermion-to-qubit mapping</h3>

In [3]:
molecule_name = "LiH" # choose one out of the molecules ['H2', 'H2_6-31g', 'LiH', 'BeH2', 'H2O', 'NH3']
mapping_name = "JW" # choose one out of ["JW","BK","Parity"]
if molecule_name == 'H2':
    basis_set = "6-31g"
else:
    basis_set = "sto3g"
num_qubits = {'H2': 8, 'LiH': 12, 'BeH2': 14, 'H2O': 14, "NH3": 16} # number of qubits in which Hamiltonian is defined
savename = molecule_name + "_molecule_" + mapping_name + "_" + basis_set
folder_OGM_settings = "OGM_probabilities/OGM_{}_{}{}.txt" # format string to fill in {molecule}x{qubit_number}x{mapping}

<h3>Import the original hamiltonian file</h3>

In [4]:
# ensure the folder exists
import os


assert os.path.isdir(folder_Hamiltonians), f"Path '{folder_Hamiltonians}' does not exist or is not a directory."
    # find folder matching molecule + basis name
available_folders = os.listdir(folder_Hamiltonians)
prefix = f"{molecule_name}_{basis_set}"
folder_name = None
for folder in available_folders:
    if folder.startswith(prefix):
        folder_name = folder
        break

assert folder_name is not None, f"File not found for molecule '{molecule_name}' and basis set '{basis_set}'."
full_folder_path = os.path.join(folder_Hamiltonians, folder_name)

    # look for encoding and energy file
available_files = os.listdir(full_folder_path)
    
    #print(f"full path to folder",full_folder_path)
file_name = None
file_energy = None
for file in available_files:
    if file.lower().startswith(mapping_name[:2].lower()) and "grouped" not in file:
        file_name = file
    elif file == "ExactEnergy.txt":
        file_energy = file

assert file_name is not None, f"File not found for encoding '{mapping_name}'."
assert file_energy is not None, "File not found for ground-state energy."
    
print(f"full path to file",os.path.join(full_folder_path, file_name))

full path to file C:\Users\Abbas\Priori\\Hamiltonians_v2\LiH_sto3g_12qubits\jw_uniformrandompaulis.txt


<h3>Loading Prior Information</h3>

In [5]:
# Loading Hamiltonian
observables, w, offset, E_GS, state = load_pauli_list(folder_Hamiltonians,molecule_name,basis_set,mapping_name,diagonalize=False)

# loading exact ground state
path_to_GS = folder_Hamiltonians + molecule_name + "_" + basis_set + "_" + str(num_qubits[molecule_name]) + "qubits\\exact_gs.txt"
with open(path_to_GS, 'r') as f:
    data_string = f.read()
state = data_string.split('\n')
state = state[:len(state)-1]
for i in range(len(state)):
    state[i] = complex(state[i])
state = np.array(state)

# Loading exact ground state energy
path_to_GS_energy = folder_Hamiltonians + molecule_name + "_" + basis_set + "_" + str(num_qubits[molecule_name]) + "qubits\\exact_gs_energy.txt"
with open(path_to_GS_energy, 'r') as f:
    data_string = f.read()
E_GS = float(data_string)

# Calculating exact covariances

sanity = False
tightest = True

if tightest:
    path_to_cov_real = folder_Hamiltonians + molecule_name + "_" + basis_set + "_" + str(num_qubits[molecule_name]) + "qubits\\cov_real_" + mapping_name + ".npy"
    os.makedirs(os.path.dirname(path_to_cov_real), exist_ok=True)
    if os.path.exists(path_to_cov_real):
        print("Loading exact covariance matrix from file...")
        #loading exact covariance matrix from file
        cov_real = np.load(path_to_cov_real)
    else:
        # Computing exact representation of full covariance matrix
        exact_covariances = covariance_matrix(state, observables)
        #print(np.round(exact_covariances,3))
        cov_real = np.ascontiguousarray(((exact_covariances + exact_covariances.conj().T).real) * 0.5, dtype=np.float64)
        # Save the computed covariance matrix to file for future runs
        np.save(path_to_cov_real, cov_real)
        print(f"Saved exact covariance matrix to {path_to_cov_real}")
if sanity:
        # Sanity check: Computing total variance using full matrix representation of Hamiltonian
        H_dot_state = apply_Hamiltonian_to_state(state, molecule_name, basis_set, mapping_name, folder_Hamiltonians)
        exp_value_H2 = np.dot(np.conjugate(H_dot_state), H_dot_state)
        exp_value_H = np.dot(np.conjugate(state), H_dot_state)
        var_H = exp_value_H2 - exp_value_H**2
        print(f"Total variance: {var_H}") # Should be zero since state is an exact eigenstate of Hamiltonian

Loading exact covariance matrix from file...


<h3>Setting Parameters for Simulation</h3>

In [6]:
Nreps = 100
eps = 0.01
delta = 0.33
Nrounds = 1000
alpha = np.max(np.abs(w))/np.min(np.abs(w)) + np.min(np.abs(w))
state_sampler = StateSampler(state)

<h3>Generating measurement scheme using Shadow_Grouping OR Priori<h3>

In [7]:
label = "OGM_NPBC" # Select the pool generation method: "SGP" for Shadow Grouping, "PP" for Priori_Pool, "OGMP" for Overlapped Grouping, "AEP" for AEQUO_Pool, "AEWP" for AEQUO_Weighted_Pool, "OGM_NPBC" for OGM without periodic boundary conditions, "STD" for steady_state_budget_allocation.
if label == "SGP":
    Nreps_SGP = 1
    Nrounds_SGP = 100000
    method_1 = Shadow_Grouping(observables,w,eps,Bernstein_bound(alpha=alpha)(), cov_real)
    estimator_1 = Energy_estimator(method_1,state_sampler,offset=offset,N_reps_exp=Nreps_SGP)
    estimator_1.reset()
    t_init_scheme_1 = time.time()
    estimator_1.propose_next_settings(Nrounds_SGP)
    t_end_scheme_1 = time.time()
    print("Elapsed time in seconds: ", t_end_scheme_1 - t_init_scheme_1)
    print("Number of different measurement settings: ", len(list(estimator_1.measurement_scheme.settings_dict.keys())))
    print("Number of hits for each observable: ", estimator_1.measurement_scheme.N_hits)

if label == "PP":
    t_init_scheme_1 = time.time()
    method_1 = Priori_Pool(observables,w,eps,Bernstein_bound(alpha=alpha)(), cov_real)
    method_1.find_setting()
    t_end_scheme_1 = time.time()
    print("Elapsed time in seconds: ", t_end_scheme_1 - t_init_scheme_1)
    print("Number of different measurement settings: ", len(list(method_1.settings_dict.keys())))
    print(method_1.N_hits)
    
if label == "OGMP":
    t_init_scheme_1 = time.time()
    # catch exception of missing data for OGM
    file = folder_OGM_settings.format(molecule_name,observables.shape[1],mapping_name.lower())
    if isfile(file):
        method_1 = OGM_Pool(observables,w,file)
        method_1.find_setting()
        t_end_scheme_1 = time.time()
        print("Elapsed time in seconds: ", t_end_scheme_1 - t_init_scheme_1)
        print("Number of different measurement settings: ", len(list(method_1.settings_dict.keys())))
        print(method_1.N_hits)
    else:
        print(f"File not found for OGM probabilities: {file}. Please run the OGM pool generation code to generate the required data.")


if label == "AEP":
    t_init_scheme_1 = time.time()
    method_1 = AEQUO_Pool(observables,w,eps,Bernstein_bound(alpha=alpha)(), cov_real)
    method_1.find_setting()
    t_end_scheme_1 = time.time()
    print("Elapsed time in seconds: ", t_end_scheme_1 - t_init_scheme_1)
    print("Number of different measurement settings: ", len(list(method_1.settings_dict.keys())))
    print(method_1.N_hits)


if label == "AEWP":
    t_init_scheme_1 = time.time()
    method_1 = AEQUO_Pool_weightbased(observables,w,eps,Bernstein_bound(alpha=alpha)(), cov_real)
    method_1.find_setting()
    t_end_scheme_1 = time.time()
    print("Elapsed time in seconds: ", t_end_scheme_1 - t_init_scheme_1)
    print("Number of different measurement settings: ", len(list(method_1.settings_dict.keys())))
    print(method_1.N_hits)


if label == "OGM_NPBC":
    t_init_scheme_1 = time.time()
    method_1 = OGM_NPBC(observables,w,eps,Bernstein_bound(alpha=alpha)(), cov_real)
    method_1.find_setting()
    t_end_scheme_1 = time.time()
    print("Elapsed time in seconds: ", t_end_scheme_1 - t_init_scheme_1)
    print("Number of different measurement settings: ", len(list(method_1.settings_dict.keys())))
    print(method_1.N_hits)

if label == "STD":
    Nreps_SGP = 1
    Nrounds_SGP = 100000
    method_1 = Shadow_Grouping(observables,w,eps,Bernstein_bound(alpha=alpha)(), cov_real)
    estimator_1 = Energy_estimator(method_1,state_sampler,offset=offset,N_reps_exp=Nreps_SGP, compat_type='qwc', qubit_connectivity=None)
    estimator_1.reset()
    t_init_scheme_1 = time.time()
    estimator_1.steady_state_budget_allocation(num_steps=Nrounds_SGP, beta=0.01, verbose=True)
    t_end_scheme_1 = time.time()
    print("Elapsed time in seconds: ", t_end_scheme_1 - t_init_scheme_1)
    print("Number of different measurement settings: ", len(list(estimator_1.measurement_scheme.settings_dict.keys())))
    print("Number of hits for each observable: ", estimator_1.measurement_scheme.N_hits)

cached settings: [array([1, 1, 3, 3, 1, 1, 2, 1, 1, 3, 2, 3]), array([3, 1, 3, 1, 3, 1, 2, 3, 3, 3, 2, 2]), array([2, 3, 1, 3, 2, 2, 2, 2, 3, 1, 3, 1]), array([1, 1, 2, 1, 1, 3, 1, 3, 2, 1, 2, 2]), array([3, 3, 3, 1, 2, 2, 1, 3, 3, 2, 3, 1]), array([3, 1, 3, 3, 3, 2, 1, 3, 1, 3, 1, 3]), array([1, 2, 2, 2, 1, 2, 1, 2, 1, 1, 3, 1]), array([3, 3, 3, 3, 3, 3, 1, 2, 3, 3, 1, 1]), array([2, 2, 2, 2, 1, 1, 3, 1, 1, 3, 3, 3]), array([1, 3, 1, 1, 3, 2, 2, 1, 3, 2, 2, 2]), array([1, 3, 2, 3, 2, 2, 3, 2, 3, 2, 1, 2]), array([3, 3, 2, 1, 3, 2, 1, 2, 2, 1, 3, 2]), array([2, 1, 3, 2, 2, 2, 3, 1, 3, 3, 2, 2]), array([3, 2, 1, 2, 2, 3, 3, 1, 3, 3, 2, 1]), array([2, 2, 2, 1, 2, 1, 2, 1, 2, 3, 3, 1]), array([2, 2, 1, 3, 3, 1, 3, 2, 2, 1, 3, 1]), array([3, 2, 1, 2, 2, 2, 1, 2, 2, 3, 1, 3]), array([1, 2, 3, 2, 1, 3, 2, 3, 3, 1, 2, 3]), array([2, 2, 1, 1, 1, 3, 3, 3, 1, 2, 2, 2]), array([1, 3, 2, 2, 2, 2, 2, 2, 1, 2, 3, 1]), array([2, 1, 2, 2, 3, 3, 2, 2, 3, 2, 1, 1]), array([1, 1, 2, 2, 3, 2, 2, 1, 2, 2, 

<h3>Trying Best_scheme_given_pool using the given pool in last cell<h3>

In [8]:
t_init_scheme_1re = time.time()
method_1re = Best_scheme_given_pool(observables,w,method_1,eps,
                                    total_rounds=Nrounds,is_overlapping=True,
                                    commutativity_type="qwc", # 'qwc', 'fc' or 'kc' 
                                    informed_allocation=True,
                                    allocation_objective="bernstein_l1", # 'variance' or 'bernstein_l1'
                                    attempt_truncation=False,
                                    rounding_strategy="largest_fraction", # 'largest_fraction' or 'marginal'
                                    md_gap_tol_rel=1e-4)
estimator_1re = Energy_estimator(method_1re,state_sampler,offset=offset,N_reps_exp=Nreps,
                                 compat_type='qwc', qubit_connectivity=None)
t_end_scheme_1re = time.time()
print("Elapsed time in seconds: ", t_end_scheme_1re - t_init_scheme_1re)
print("Number of different measurement settings: ", len(list(estimator_1re.measurement_scheme.settings_dict.keys())))
print(estimator_1re.measurement_scheme.N_hits)
epsilonbernstein_1re , _ = get_epsilon_Bernstein_no_restricted_validity(delta, estimator_1re.measurement_scheme.N_hits, w, estimator_1re.measurement_scheme.settings_dict)
print(f"Epsilon Bernstein for RE: {epsilonbernstein_1re}")
epsilonchebyshevtighter_1re = get_epsilon_Chebyshev_scalar_tighter_numba(delta, estimator_1re.measurement_scheme.N_hits, estimator_1re.measurement_scheme.N_hits_pairs, w)
print(f"Epsilon Chebyshev (tighter) for RE: {epsilonchebyshevtighter_1re}")
epsilonchychev_tightest_1re = get_epsilon_Chebyshev_scalar_tightest_numba(delta, estimator_1re.measurement_scheme.N_hits, estimator_1re.measurement_scheme.N_hits_pairs, w, cov_real)
print(f"Epsilon Chebyshev (tightest) for RE: {epsilonchychev_tightest_1re}")
estimator_1re.clear_outcomes()
estimator_1re.state = StateSampler(state)
estimator_1re.measure_and_get_running_avgs()
energy_estimates = estimator_1re.get_energy()
rmse, rmse_se, (ci_lo, ci_hi), boot_vals = bootstrap_rmse(energy_estimates, E_GS, n_boot=20000, ci=0.67)
print(f"Results for RE | RMSE: {rmse:.6f} | Std: {rmse_se:.6f}")

Updated N_reps_exp to 100. Cleared outcomes using clear_outcomes().
Elapsed time in seconds:  16.58634114265442
Number of different measurement settings:  153
[  5   9   7   5 154   4   6  22 150   7   6   4  40  97   6   7  18   9
   4  78 131  26  57  30   6   4  53  13  46 108  48  41  30   7  19  88
  97  25 174  11   7  18 124  41   4  41   4 128  14   6   4  97   6   6
  11  97   8  21  15   7   5   5   4   7 110  85  21  33   6   8  30 134
  10  11 125  19  24  14   5  47  15   7  29  12   7  18   6  14   6  31
  13  13  89  74   6  13  17 127 102   5   5   7  15  12  13  63  14  11
 134   5  57  30   6   7   4 117  35   9   7   6   9  91  15  33  42   7
  14   7  12  15   7  90   7  41  14  12   7  10   7 107  22  81   6 112
   4 133   7   8  52  10  25  11  80   6  88  13   6   7  22   7  38  41
  13  24 122   7   7  39   5   8   7 135  84 156  58 115  25 102  35  13
  28  72  34  39 145 159  23  46  15  11 122 107 114  12   8   7 129   7
  11  16  88  12   8  18   6 115   9  

[Qibo 0.1.11|INFO|2026-06-02 19:09:01]: Using numpy backend on /CPU:0


Epsilon Chebyshev (tightest) for RE: 12.607617950682343
Results for RE | RMSE: 14.823558 | Std: 0.771822


<h3>Comparing with ShadowGrouping<h3>

In [9]:
Test = "Y" # If you want to compare the results with the ShadowGrouping, set Test to "Y" otherwise set it to "N"
if Test == "Y":
    method = Shadow_Grouping(observables,w,eps,Bernstein_bound(alpha=alpha)(), cov_real)
    estimator = Energy_estimator(method,state_sampler,offset=offset,N_reps_exp=Nreps)
    estimator.reset()
    t_init_scheme = time.time()
    estimator.propose_next_settings(Nrounds)
    t_end_scheme = time.time()
    print("Elapsed time in seconds: ", t_end_scheme - t_init_scheme)
    print("Number of different measurement settings: ", len(list(estimator.measurement_scheme.settings_dict.keys())))
    #print("Keys in settings_dict: ", list(estimator.measurement_scheme.settings_dict.keys()))    
    print("Number of hits for each observable: ", estimator.measurement_scheme.N_hits)
    #print("Indices of settings: ", estimator.measurement_scheme.setting_indices)
    epsilonbernstein , _ = get_epsilon_Bernstein_no_restricted_validity(delta, estimator.measurement_scheme.N_hits, w, estimator.measurement_scheme.settings_dict)
    print(f"Epsilon Bernstein for Shadow Grouping: {epsilonbernstein}")
    epsilonchebyshevtighter = get_epsilon_Chebyshev_scalar_tighter_numba(delta, estimator.measurement_scheme.N_hits, estimator.measurement_scheme.N_hits_pairs, w)
    print(f"Epsilon Chebyshev (tighter) for Shadow Grouping: {epsilonchebyshevtighter}")
    epsilonchychev_tightest = get_epsilon_Chebyshev_scalar_tightest_numba(delta, estimator.measurement_scheme.N_hits, estimator.measurement_scheme.N_hits_pairs, w, cov_real)
    print(f"Epsilon Chebyshev (tightest) for Shadow Grouping: {epsilonchychev_tightest}")
    estimator.clear_outcomes()
    estimator.state = StateSampler(state)
    estimator.measure_and_get_running_avgs()
    energy_estimates = estimator.get_energy()
    rmse, rmse_se, (ci_lo, ci_hi), boot_vals = bootstrap_rmse(energy_estimates, E_GS, n_boot=20000, ci=0.67)
    print(f"Results for ShadowGrouping | RMSE: {rmse:.6f} | Std: {rmse_se:.6f}")
if Test == "N":
    raise SystemExit("Execution stopped at this cell.")

Updated N_reps_exp to 100. Cleared outcomes using clear_outcomes().
Elapsed time in seconds:  6.848633289337158
Number of different measurement settings:  650
Number of hits for each observable:  [ 10  10  27  10 117  10   9  10 145  10  10   9  28 128  10  21  13  10
  10 112 155  28  42  20  10  10  57  12  41 104  39  42  34  11  15 116
 108  14 138  11  10  13 105  48  10  44  10 125  10  10  10 101  10  12
  10  99  10  22  16  10  10  10  10  11  85  77  10  40  10  10  16 137
  17  12 138  10  36  15  10  40  10  10  22  10  10  14  10  18  11  33
  12  10 112 106  10  12  17 102 123  10  10  10  14  11  14  50  10  13
 137  10  36  11   9   9   9 107  44   9   9   9  11  95  17  29  34   9
  14  11  15  15  12  85  18  36  21   9   9  10  10 116  44 101   9 106
   9 131   9   9  51  16  25  10 114   9 110  34   9  10  11   9  55  32
  15  13 119  10   9  20  11   9  10 110 126 130  71 120  32 105  39  19
  15 103  38  27 132 110  24  42  23   9 115 116 118  15  10  10 130  10
 

In [10]:
# Debug the failing QWC setting reconstruction
ms = estimator_1re.measurement_scheme
print('num settings:', len(ms.settings_buffer))
for setting_token in ms.settings_buffer:
    obs_ids = __import__('shadowgrouping_v2.helper_functions', fromlist=['decode_setting_token']).decode_setting_token(setting_token).astype(int)
    setting_int = __import__('numpy').zeros(ms.num_qubits, dtype=int)
    for oid in obs_ids:
        o = ms.obs[oid]
        non_id = (o != 0)
        fill = non_id & (setting_int == 0)
        setting_int[fill] = o[fill]
    ok = __import__('shadowgrouping_v2.shadowgrouping_my_dev.qubit_wise_commutativity', fromlist=['hit_by_batch_numba']).hit_by_batch_numba(ms.obs[obs_ids], setting_int)
    if not ok.all():
        print('bad token:', setting_token)
        print('obs ids:', obs_ids)
        print('setting int:', setting_int)
        print('bad obs:', ms.obs[obs_ids][~ok])
        break
else:
    print('all settings are QWC-consistent')

num settings: 0
all settings are QWC-consistent
